# 4.2 State, Nodes, Edges, Conditional Edges

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Learning objectives

- Define graph state as a `TypedDict` and explain the default rule: a returned value replaces the field.
- Write nodes as plain functions that take state and return partial updates.
- Connect nodes with fixed edges (`START`, `END`) and route with conditional edges.
- Use reducers (`Annotated[..., operator.add]`, `add_messages`) to merge updates instead of overwriting.
- Use `MessagesState` as the shortcut for conversation state, compile a graph, invoke it, and read the execution trace.

## A graph is a state machine

Strip away the marketing and a LangGraph graph is a state machine with four pieces:

1. **State**: one shared board, declared as a `TypedDict`, that every node reads and writes.
2. **Nodes**: plain functions. Each takes the current state, does one unit of work, and returns the changes.
3. **Edges**: fixed roads between nodes. Conditional edges are roads with a router that picks the destination from the state.
4. **A compiled graph**: the builder turned into something you can `invoke`.

No magic. A node is a function, an edge is "run this next", and state is how nodes talk to each other. The whole lesson is the details of those four pieces, and every detail exists to answer one question: when a node returns an update, how does it merge into the board?

## State: the shared board

State is declared as a `TypedDict`. Each field is one channel on the board:

In [ ]:
from typing_extensions import TypedDict

class SupportState(TypedDict):
    question: str
    category: str
    answer: str

The default merge rule is brutally simple: **a node's returned value replaces the field's current value.** Whatever the node returns for `answer` becomes the new `answer`; fields the node does not return keep their old values. Watch what that default does to a list:

In [ ]:
class PlainState(TypedDict):
    messages: list

builder = StateGraph(PlainState)
builder.add_node("a", lambda state: {"messages": ["from-a"]})
builder.add_edge(START, "a")
builder.add_edge("a", END)

graph = builder.compile()
result = graph.invoke({"messages": ["seed"]})
print(result["messages"])   # ['from-a']  -- the seed message is gone

The node's `["from-a"]` replaced `["seed"]`, not appended to it. This is the single most common beginner bug in LangGraph: people assume lists accumulate, and the framework quietly discards everything but the last write. Nothing errors. The history just vanishes. Reducers are the fix.

## Reducers: how updates merge

A reducer is attached to a field with `Annotated`, and it decides how a node's update combines with the existing value:

In [ ]:
import operator
from typing import Annotated

class SupportState(TypedDict):
    question: str
    category: str
    answer: str
    audit: Annotated[list, operator.add]   # append updates, do not replace

Now every node that returns `{"audit": [...]}` has its list appended to the running list instead of replacing it. This is exactly the parallel-workers case from lesson 4.1: two branches writing `audit` merge instead of clobbering each other. For non-list fields you can write your own reducer as a plain two-argument function; a retry counter, for example, can be `Annotated[int, lambda old, new: old + new]`, so each node increments rather than overwrites.

For message lists, LangGraph ships a purpose-built reducer, `add_messages`, which appends messages and handles the bookkeeping message objects need (matching by id, removing messages marked for deletion):

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

Every node returns `{"messages": [new_message]}` and the conversation grows. If you need to delete a message, return it wrapped for removal and the reducer drops it from the list. The principle is the same everywhere: **replacement is the default; if you want anything else, say so in the schema.**

## Nodes: plain functions

A node is a function that takes the state and returns a dictionary of updates. That is the whole contract:

In [ ]:
def classify(state: SupportState) -> dict:
    q = state["question"].lower()
    category = "billing" if ("refund" in q or "charged" in q) else "technical"
    return {"category": category, "audit": ["classified"]}

Three rules worth internalising:

- **Return updates, not the whole state.** Only the fields that changed go in the dictionary. Fields you omit keep their values.
- **Do not mutate the state you were given.** Nodes should compute new values and return them. Mutating `state` in place works in a quick demo and corrupts real runs, because checkpointing and parallel branches assume each node's input is untouched.
- **A node can do anything.** Call an LLM, query a database, hit a REST API: the graph treats a node with an LLM inside exactly like a node without one. That is the whole answer to "but where is the intelligence": the intelligence lives in the nodes that call models; the graph is the wiring that decides which node runs when, and it is wiring you can see.

## Edges: fixed roads

Edges connect nodes. `START` marks where a run begins, `END` marks where it stops, and both are imported from `langgraph.graph`:

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)

builder.add_edge(START, "classify")
builder.add_edge("billing", END)
builder.add_edge("technical", END)

Old tutorials (pre-1.0 LangGraph) used `set_entry_point("classify")` and the string literals `"__start__"` / `"__end__"`. The current API uses `add_edge(START, ...)` and the `START` / `END` constants. If a tutorial's wiring code fails on the pinned versions, that is the first thing to check.

Every node needs a way out: an edge to another node or to `END`. A node with no outgoing edge is a dead end that will fail, not a silent stop. Say it with `END` and mean it.

## Conditional edges: routing

A conditional edge calls a router, a plain function of state, and follows its answer:

In [ ]:
def route(state: SupportState) -> str:
    return state["category"]          # "billing" or "technical"

builder.add_conditional_edges(
    "classify",
    route,
    {"billing": "billing", "technical": "technical"},
)

The mapping dict translates the router's answer into a destination node. If the router returns a name that is not in the mapping and not a node name, the run fails; misspelt destinations are compile-time errors, which is the framework doing you a favour.

The router is the most testable part of any graph. No model, no network:

In [ ]:
assert route({"category": "billing"}) == "billing"
assert route({"category": "technical"}) == "technical"

Routing is where agent bugs live, and this is the one piece you can pin down with a unit test. When a run goes somewhere unexpected, read the router and the state it read before blaming the model.

## The router demo, running

The full graph from this lesson, no LLM required. Save and run it:

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class SupportState(TypedDict):
    question: str
    category: str
    answer: str
    audit: Annotated[list, operator.add]

def classify(state: SupportState) -> dict:
    q = state["question"].lower()
    category = "billing" if ("refund" in q or "charged" in q) else "technical"
    return {"category": category, "audit": ["classified"]}

def billing_node(state: SupportState) -> dict:
    return {"answer": "Billing team will help you.", "audit": ["billing"]}

def technical_node(state: SupportState) -> dict:
    return {"answer": "Technical team will help you.", "audit": ["technical"]}

def route(state: SupportState) -> str:
    return state["category"]

builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route,
                             {"billing": "billing", "technical": "technical"})
builder.add_edge("billing", END)
builder.add_edge("technical", END)

graph = builder.compile()

for question in ["I was charged twice, need a refund", "My app crashes on login"]:
    result = graph.invoke({"question": question, "audit": []})
    print(result["category"], "->", result["answer"], "| audit:", result["audit"])

Expected output:

In [ ]:
billing -> Billing team will help you. | audit: ['classified', 'billing']
technical -> Technical team will help you. | audit: ['classified', 'technical']

Notice the `invoke` input initialises every field the run will touch: `question` for the classifier, `audit: []` so the `operator.add` reducer has a list to append to. If you omit `audit`, the reducer has nothing to add to and the run fails. Initialise your channels.

## MessagesState: the shortcut

Conversation state (a message list plus a few extra fields) is common enough that LangGraph ships it ready-made:

In [ ]:
from langgraph.graph import MessagesState

class AgentState(MessagesState):
    category: str        # your extra fields sit alongside messages

`MessagesState` declares `messages: Annotated[list[BaseMessage], add_messages]` for you, so every node that returns `{"messages": [...]}` appends to the conversation. Subclass it and add whatever else the graph needs; the reducer is already attached. The standard agent graph (lesson 2.3's `create_agent` internals, and lesson 4.3's rebuild) runs on this shape: an LLM node appends its reply, a tools node appends tool results, and a conditional edge loops until there are no more tool calls.

## Reading the trace

When a graph misbehaves, do not stare at the final output. Stream the run with `stream_mode="updates"` and read what each node returned:

In [ ]:
for update in graph.stream({"question": "refund please", "audit": []},
                           stream_mode="updates"):
    print(update)

In [ ]:
{'classify': {'category': 'billing', 'audit': ['classified']}}
{'billing': {'answer': 'Billing team will help you.', 'audit': ['billing']}}

Each chunk names the node that ran and the exact update it returned. Compare that against what you expected: the wrong destination means the router or the state it read; the wrong update means the node. This is the debugging habit for the whole module. Final-state inspection tells you the run is wrong; the update trace tells you where.

A note on the `Send` API, briefly: conditional edges normally route to one fixed node per branch, but sometimes the number of parallel tasks is only known at runtime (a planner emits five research questions, or fifty). A router can return a list of `Send("node_name", input_dict)` objects, one per task, and LangGraph runs each as a parallel invocation whose results merge through the reducers. Static fan-out (one edge per branch, known at build time) is enough for this lesson; `Send` is the dynamic version, and you will meet it in the map-reduce lesson. Cap the list before you build the sends: a model-generated fan-out with no hard budget is a cost incident waiting to happen.

## What can go wrong

- **Assuming lists accumulate.** The default merge is replacement. If a message history or findings list keeps losing everything but the last write, the schema is missing its reducer. Attach `operator.add` or `add_messages` in the `TypedDict`, not in the node.
- **Router returns an unknown name.** A typo in the mapping or a router branch you forgot maps to a runtime failure. Misspelt node names in `add_edge` fail at compile time; treat either as a prompt to check the names, and keep the router's possible answers in one obvious place.
- **Forgetting the START edge.** A graph with nodes and edges but no `add_edge(START, ...)` compiles fine and does nothing. If `invoke` returns the input unchanged, check the entry edge first.
- **Reading state a node has not set yet.** The router reads `state["category"]`, so `classify` must run first and must return it. If a node reads a key that was never set, you get a `KeyError`, not an empty value. Initialise inputs at `invoke` and order your edges so writers run before readers.
- **Mutating state inside a node.** Appending to `state["audit"]` directly instead of returning `{"audit": [...]}` works once and breaks under checkpointing and parallelism, where the framework assumes inputs are immutable. Return updates; never mutate.
- **Parallel branches sharing a plain field.** Two nodes returning the same non-reducer field means last write wins and one branch's output silently disappears. Any field two branches write needs a reducer, or the branches need separate fields.
- **Personal data in state, persisted by default.** The moment you add a checkpointer (lesson 4.4 onward), every state snapshot is stored, and state often contains user messages. For an EU audience this is a GDPR design point from day one: keep personal data out of state fields you do not need, and know where your snapshots live before the first production run.

## Check your understanding

**1. A node returns `{"messages": ["hello"]}` and the previous state had `messages: ["hi"]`. What is the final value, with and without a reducer?**
Without a reducer, the default rule is replacement: `["hello"]`, and `"hi"` is lost. With `Annotated[list, operator.add]` (or `add_messages` for message objects), the update appends: `["hi", "hello"]`.

**2. What is a router, and why is it the easiest part of a graph to test?**
A router is a plain function of state passed to `add_conditional_edges`; it returns the name of the next node (or a value the mapping translates into one). It contains no LLM call and no network access, so you can test it with a plain dictionary and an assertion: `assert route({"category": "billing"}) == "billing"`. Routing is where agent bugs live, and the router is the one piece you can pin down without running the model.

**3. Your graph compiles, `invoke` returns the input unchanged, and no node ran. What is the most likely cause?**
A missing entry edge: there is no `add_edge(START, "first_node")`, so the run starts and immediately has nowhere to go. Check that `START` points at your first node and that every node has an outgoing edge or an edge to `END`.

**4. Two nodes run in parallel and both return `{"findings": [...]}`. The final state only has one node's findings. Explain the bug and the two possible fixes.**
The `findings` field has no reducer, so the default replacement rule applies and the second write overwrites the first: last write wins. Fix one: attach a reducer in the schema, `findings: Annotated[list, operator.add]`, so parallel updates append. Fix two: give each branch its own field (e.g. `findings_a`, `findings_b`) and merge them in a downstream node.

**5. When should you reach for `MessagesState` instead of declaring your own `TypedDict` from scratch, and what does it give you?**
When the graph's state is a conversation plus a few extra fields. `MessagesState` pre-declares `messages: Annotated[list[BaseMessage], add_messages]`, so every node that returns `{"messages": [...]}` appends to the history correctly. You subclass it and add only your extra fields (a category, a retry counter), and the message-list reducer is already attached.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.